# SKKB — Data Preparation

JIRA: **GCAI-3092** — LLM-as-a-Judge evaluation for the SK (Slovak) Knowledge-Base (KB).

This notebook reads the raw MLflow traces JSON for the 513-case SK KB test set and writes a flat Delta table that the judge-run notebook then consumes via `hg_ds_evals.evals.run_evals.run_experiment`.

**Input:** `ds-evals/input/kb_sk/traces_SK_513.json`
**Output:** Delta table `{catalog}.{schema}.dts_eval_skkb_exp_001`

Per-trace extractions and the column contract are documented in [`ds-evals/docs/skkb/skkb_traces_schema.md`](../../docs/skkb/skkb_traces_schema.md).

## Environment

### Imports

In [0]:
# !pip install -qe /Workspace/Repos/Shared_HeyGeorge/hg-ds-evals
# !pip install -qe /Workspace/Repos/Shared_HeyGeorge/hey-george-ds/ds_common
!pip install openai -q

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
RUN_IN_DBX:bool = True

In [0]:
import config_nbs
config_nbs.add_local_paths()

import hashlib
import json
import re
from collections import Counter
from typing import Any, Iterable

import pandas as pd
pd.set_option("display.max_colwidth", None)
from pyspark.sql import SparkSession

if RUN_IN_DBX:
    spark = SparkSession.builder.getOrCreate()


Path "/Workspace/Users/sg7cb@s-mxs.net/hg-ds-evals/" added!
Path "/Workspace/Users/sg7cb@s-mxs.net/hg-ds-evals/experiments/skkb/" added!


### Parameters


In [0]:
%skip
!ls -la ../input/

In [0]:
# ── Parameters ────────────────────────────────────────────────
if RUN_IN_DBX:
    # raw SKKB traces JSON
    # SKKB_JSON_PATH:str = "/Workspace/Repos/Shared_HeyGeorge/ds-evals/input/kb_sk/traces_SK_513.json"
    SKKB_JSON_PATH:str = "../input/traces_SK_513.json"
else:
    SKKB_JSON_PATH:str = "../../input/kb_sk/traces_SK_513.json"

# Target Delta table (must match the YAML: dataset.input_catalog + dataset.input_dataset + dataset.input_schema)
DBX_CATALOG:str = "prod_aut_chat00_lab_catalog"
DBX_SCHEMA:str = "private_uc0115_aix_db"
DBX_TABLE:str = "dts_eval_skkb_exp_001"

# If true, drop rows where any of user_query / bot_response / expected_response is empty.
DROP_INCOMPLETE:bool = True


## Parser helpers

The MLflow trace-metadata `mlflow.traceInputs` / `mlflow.traceOutputs` and the `request_preview` / `response_preview` fields are **truncated previews**. The full bot response must be reconstructed from the spans.

See `docs/skkb/skkb_traces_schema.md` for the details and derivation.


In [0]:
AI_TYPE_STRINGS = {"ai", "AIMessage", "AIMessageChunk"}
ENUM_RE = re.compile(r"(?m)^([A-Z][A-Z0-9_@-]*(?:\s*\+\s*[A-Z][A-Z0-9_@-]*)?)(?=:)")

# The agent system prompt ends with a timestamp that changes per trace.
# Strip it before hashing so we can detect prompt-text drift independently
# of clock drift.
_TS_TAIL_RE = re.compile(r"Current date and time:.*$", re.DOTALL)


def _walk_dicts(obj: Any) -> Iterable[dict]:
    """Recursively yield every dict nested inside a JSON-like structure."""
    if isinstance(obj, dict):
        yield obj
        for v in obj.values():
            yield from _walk_dicts(v)
    elif isinstance(obj, list):
        for v in obj:
            yield from _walk_dicts(v)


def _build_span_children(spans: list[dict]) -> dict[str, list[dict]]:
    kids: dict[str, list[dict]] = {}
    for s in spans:
        p = s.get("parent_span_id")
        if p:
            kids.setdefault(p, []).append(s)
    return kids


def _find_descendant_by_name(root: dict, kids: dict[str, list[dict]], name: str) -> dict | None:
    for c in kids.get(root.get("span_id"), []):
        if c.get("name") == name:
            return c
        deeper = _find_descendant_by_name(c, kids, name)
        if deeper is not None:
            return deeper
    return None


def _parse_attr_json(attrs: dict, key: str):
    raw = attrs.get(key, "")
    if not raw:
        return None
    if isinstance(raw, str):
        try:
            return json.loads(raw)
        except json.JSONDecodeError:
            return None
    return raw


def _md5_short(s: str) -> str:
    """Stable short hash (10 hex chars) over a (possibly empty) string."""
    return hashlib.md5((s or "").encode("utf-8", errors="ignore")).hexdigest()[:10]


def _get_agent_chat_system_prompt(spans: list[dict], kids: dict, agent_name: str) -> str:
    """Return the system-message content of the ChatDatabricks call owned by
    the named agent span (`main_agent` or `daily_banking_agent`).

    For `daily_banking_agent` the ChatDatabricks we care about is under the
    `llm` child node, not the agent's own ChatDatabricks (which only gets a
    short routing prompt). We take the first ChatDatabricks descendant
    whose system prompt is non-empty.
    """
    agent = next((s for s in spans if s.get("name") == agent_name), None)
    if agent is None:
        return ""
    # Find any ChatDatabricks in the subtree with a non-trivial system message.
    stack = [agent]
    while stack:
        cur = stack.pop()
        if cur.get("name") == "ChatDatabricks":
            si = _parse_attr_json(cur.get("attributes") or {}, "mlflow.spanInputs")
            if isinstance(si, list) and si and isinstance(si[0], list):
                for m in si[0]:
                    if isinstance(m, dict) and (m.get("role") == "system" or m.get("type") == "system"):
                        c = m.get("content", "")
                        if isinstance(c, str) and c.strip():
                            return c
        stack.extend(kids.get(cur.get("span_id"), []))
    return ""


def extract_agent_prompt_hashes(spans: list[dict], kids: dict) -> dict:
    """Fingerprint the main_agent + daily_banking_agent system prompts.

    The canonical prompt text (minus the trailing timestamp) is stable across
    the dataset — see configs/skkb/agent_context/*. Hashes let us confirm
    per-row that the live prompt still matches the checked-in canonical copy,
    and would surface any drift. An empty-string hash means the agent was
    never invoked in this trace (e.g. main_agent handled the turn alone).
    """
    main_raw = _get_agent_chat_system_prompt(spans, kids, "main_agent")
    dba_raw = _get_agent_chat_system_prompt(spans, kids, "daily_banking_agent")
    main_stripped = _TS_TAIL_RE.sub("", main_raw).rstrip()
    dba_stripped = _TS_TAIL_RE.sub("", dba_raw).rstrip()
    return {
        "main_agent_prompt_hash": _md5_short(main_stripped),
        "daily_banking_agent_prompt_hash": _md5_short(dba_stripped),
    }


def extract_final_ai_content(spans: list[dict]) -> str:
    """Find the longest AI-typed message content across all span outputs."""
    best = ""
    for span in spans:
        parsed = _parse_attr_json(span.get("attributes") or {}, "mlflow.spanOutputs")
        if parsed is None:
            continue
        for node in _walk_dicts(parsed):
            t = node.get("type")
            if t not in AI_TYPE_STRINGS and str(t).lower() != "ai":
                continue
            content = node.get("content")
            if not isinstance(content, str) or not content.strip():
                continue
            meta = node.get("response_metadata") or {}
            finish = meta.get("finish_reason")
            if finish is not None and finish != "stop":
                continue
            if len(content) > len(best):
                best = content
    return best


def extract_kb_context(spans: list[dict]) -> str:
    """Return the `content` string from the first `knowledge_search` span's outputs."""
    for span in spans:
        if span.get("name") != "knowledge_search":
            continue
        parsed = _parse_attr_json(span.get("attributes") or {}, "mlflow.spanOutputs")
        if isinstance(parsed, dict):
            c = parsed.get("content")
            if isinstance(c, str) and c.strip():
                return c
    return ""


def extract_reranked_enum_ids(kb_context: str) -> list[str]:
    """Pull enum IDs from KB context block headers, deduplicated."""
    seen, out = set(), []
    for m in ENUM_RE.findall(kb_context or ""):
        if m not in seen:
            seen.add(m)
            out.append(m)
    return out


def extract_retrieved_candidates(spans: list[dict]) -> tuple[str, list[str]]:
    """Extract the pre-rerank candidate pool from the `retrieve` span.

    Returns (formatted_string, id_list). The formatted string is what the
    judge sees — one block per candidate in rank order:

        SAVING@ACCOUNT_MOVEMENTS: <description>

        SAVING@KIDS_DISPOSING_PERSON: <description>

        ...

    Returns ("", []) when no `retrieve` span is present (fallback path).
    """
    for span in spans:
        if span.get("name") != "retrieve":
            continue
        parsed = _parse_attr_json(span.get("attributes") or {}, "mlflow.spanOutputs")
        if not isinstance(parsed, dict):
            continue
        docs = parsed.get("retrieved_documents") or []
        if not isinstance(docs, list):
            continue
        ids: list[str] = []
        blocks: list[str] = []
        for d in docs:
            if not isinstance(d, dict):
                continue
            frag = d.get("fragment") or {}
            eid = frag.get("id")
            desc = frag.get("description", "")
            if not isinstance(eid, str) or not eid:
                continue
            ids.append(eid)
            desc_s = desc if isinstance(desc, str) else json.dumps(desc, ensure_ascii=False)
            blocks.append(f"{eid}: {desc_s}")
        return ("\n\n".join(blocks), ids)
    return ("", [])


def extract_reranker_info(spans: list[dict], kids: dict[str, list[dict]]) -> dict:
    """Return the inputs/outputs of the LLM reranker's ChatDatabricks call."""
    empty = {
        "reranker_system_prompt": "",
        "reranker_user_prompt": "",
        "reranker_model": "",
        "reranker_token_usage": "",
        "reranker_selected_ids": "[]",
        "_reranker_span_present": False,
        "_reranker_selected_ids_found": False,
    }
    rerank = next((s for s in spans if s.get("name") == "rerank"), None)
    if rerank is None:
        return empty
    cd = _find_descendant_by_name(rerank, kids, "ChatDatabricks")
    if cd is None:
        return {**empty, "_reranker_span_present": True}
    attrs = cd.get("attributes") or {}

    system_prompt = ""
    user_prompt = ""
    si = _parse_attr_json(attrs, "mlflow.spanInputs")
    msgs = None
    if isinstance(si, list) and si and isinstance(si[0], list):
        msgs = si[0]
    elif isinstance(si, list):
        msgs = si
    if msgs:
        for m in msgs:
            if not isinstance(m, dict):
                continue
            role = m.get("role") or m.get("type")
            content = m.get("content", "")
            if not isinstance(content, str):
                content = json.dumps(content, ensure_ascii=False)
            if role == "system":
                system_prompt = content
            elif role in ("user", "human"):
                user_prompt = content

    model_raw = attrs.get("mlflow.llm.model", "")
    try:
        model = json.loads(model_raw) if isinstance(model_raw, str) and model_raw.strip().startswith('"') else model_raw
    except json.JSONDecodeError:
        model = model_raw
    if not isinstance(model, str):
        model = str(model)

    token_usage_raw = attrs.get("mlflow.chat.tokenUsage", "")
    if not isinstance(token_usage_raw, str):
        token_usage_raw = json.dumps(token_usage_raw, ensure_ascii=False)

    selected_ids: list[str] = []
    selected_ids_found = False
    so = _parse_attr_json(attrs, "mlflow.spanOutputs")
    if so is not None:
        for node in _walk_dicts(so):
            tcs = node.get("tool_calls")
            if not isinstance(tcs, list):
                continue
            for tc in tcs:
                if not isinstance(tc, dict):
                    continue
                fn = tc.get("function") or {}
                if fn.get("name") != "_RerankResponse":
                    continue
                args_raw = fn.get("arguments", "")
                if not args_raw:
                    continue
                try:
                    args = json.loads(args_raw) if isinstance(args_raw, str) else args_raw
                except json.JSONDecodeError:
                    continue
                if isinstance(args, dict) and isinstance(args.get("selected_ids"), list):
                    selected_ids = [x for x in args["selected_ids"] if isinstance(x, str)]
                    selected_ids_found = True
                    break
            if selected_ids_found:
                break

    return {
        "reranker_system_prompt": system_prompt,
        "reranker_user_prompt": user_prompt,
        "reranker_model": model,
        "reranker_token_usage": token_usage_raw,
        "reranker_selected_ids": json.dumps(selected_ids, ensure_ascii=False),
        "_reranker_span_present": True,
        "_reranker_selected_ids_found": selected_ids_found,
    }


def extract_kb_version(spans: list[dict]) -> str:
    """Return the KB snapshot id (e.g. ``KB_GAI_SK_SK_2026-03-17_17h26``)
    the retriever was asked to search against.

    Preferred source: ``retrieve.spanInputs.knowledge_group_names[0]`` —
    this is the exact list the retriever took as input. Fallback:
    ``retrieve.spanOutputs.retrieved_documents[0].group_name`` (useful
    if the span happens to drop the input echo). Returns ``""`` for
    fallback-path traces with no ``retrieve`` span.
    """
    for span in spans:
        if span.get("name") != "retrieve":
            continue
        attrs = span.get("attributes") or {}
        si = _parse_attr_json(attrs, "mlflow.spanInputs")
        if isinstance(si, dict):
            names = si.get("knowledge_group_names") or []
            if isinstance(names, list) and names and isinstance(names[0], str):
                return names[0]
        so = _parse_attr_json(attrs, "mlflow.spanOutputs")
        if isinstance(so, dict):
            docs = so.get("retrieved_documents") or []
            if isinstance(docs, list):
                for d in docs:
                    if isinstance(d, dict):
                        gn = d.get("group_name")
                        if isinstance(gn, str) and gn:
                            return gn
        return ""
    return ""


def extract_agent_and_tool_calls(spans: list[dict]) -> dict:
    """Return the ordered list of LangGraph nodes visited and the tool
    calls made during the trace.

    - `agents_called` preserves first-seen order of `langgraph_node` values
      read from each span's `attributes.metadata` (JSON-stringified). In
      this dataset the values are a subset of
      {main_agent, daily_banking_agent, llm, tools, retrieve, rerank}.
    - `tools_called` is a list of {name, inputs} dicts, one per span where
      `attributes.mlflow.spanType == "TOOL"`. In this dataset only
      `knowledge_search` appears, but the extractor is general.
    """
    agents_seen: list[str] = []
    seen_set: set[str] = set()
    tools: list[dict] = []

    for span in spans:
        attrs = span.get("attributes") or {}

        md = _parse_attr_json(attrs, "metadata")
        if isinstance(md, dict):
            node = md.get("langgraph_node")
            if isinstance(node, str) and node and node not in seen_set:
                seen_set.add(node)
                agents_seen.append(node)

        st_raw = attrs.get("mlflow.spanType", "")
        if isinstance(st_raw, str) and st_raw.startswith('"'):
            try:
                st = json.loads(st_raw)
            except json.JSONDecodeError:
                st = st_raw
        else:
            st = st_raw
        if st == "TOOL":
            tools.append({
                "name": span.get("name"),
                "inputs": _parse_attr_json(attrs, "mlflow.spanInputs"),
            })

    agents_only = [n for n in agents_seen if n in KNOWN_AGENT_NAMES]
    last_agent = agents_only[-1] if agents_only else ""

    return {
        "agents_called": agents_seen,
        "agents_only": agents_only,
        "last_agent": last_agent,
        "tools_called": tools,
    }


# Tool-name classification for query_scope. knowledge_search is the KB tool;
# any tool starting with `mock_banking_` is a banking-data tool (out of scope
# for the KB judge rubric).
KB_TOOL_NAMES = {"knowledge_search"}
BANKING_DATA_TOOL_PREFIX = "mock_banking_"
HG_INVEST_AGENT_NAME = "hg-invest-phase2"
# TODO: confirm exact prefix once hg-invest tools land in a future PR.
HG_INVEST_DATA_TOOL_PREFIX = "mock_invest_"
# `agents_called` actually carries the entire LangGraph traversal (supervisor +
# sub-agents + internal nodes like `llm`, `tools`, `retrieve`, `prune`, `rerank`).
# KNOWN_AGENT_NAMES filters that down to actual agent slugs.
KNOWN_AGENT_NAMES = frozenset({"main_agent", "daily_banking_agent", HG_INVEST_AGENT_NAME})


def classify_query_scope(agents_called: list[str], tools_called: list[dict]) -> str:
    """Programmatically classify the routing outcome of a single trace.

    Daily-banking branch returns one of:
        "kb"              — DBA invoked AND knowledge_search was called.
        "banking_data"    — DBA invoked + mock_banking_* tools, no KB.
        "dba_no_tools"    — DBA reached but no tools called (Rule 1 violation).
        "other_tools"     — DBA reached + unrecognised tool mix.
    HG-invest branch returns the parallel fan-out:
        "hg_invest_kb", "hg_invest_mock_tool",
        "hg_invest_no_tools", "hg_invest_other_tools".
    Otherwise:
        "main_agent"      — neither sub-agent invoked.
    """
    tool_names = {t.get("name", "") for t in tools_called or []}
    agents = agents_called or []
    reached_dba = "daily_banking_agent" in agents
    reached_hg_invest = HG_INVEST_AGENT_NAME in agents

    if reached_dba:
        if not tool_names:
            return "dba_no_tools"
        if tool_names & KB_TOOL_NAMES:
            return "kb"
        if any(n.startswith(BANKING_DATA_TOOL_PREFIX) for n in tool_names):
            return "banking_data"
        return "other_tools"

    if reached_hg_invest:
        if not tool_names:
            return "hg_invest_no_tools"
        if tool_names & KB_TOOL_NAMES:
            return "hg_invest_kb"
        if any(n.startswith(HG_INVEST_DATA_TOOL_PREFIX) for n in tool_names):
            return "hg_invest_mock_tool"
        return "hg_invest_other_tools"

    return "main_agent"


# ── Deterministic trace-shape invariants ──────────────────────────────
# Each rule inspects only `agents_called`, `tools_called` (normalised to a
# set of tool names), and `query_scope`, and returns True when the trace
# *violates* the invariant. Violations are collected into the
# `trace_invariant_violations` column (list of rule IDs), which flags routing
# regressions before the LLM judge even runs. These are pure structural
# checks on the MLflow trace shape — no LLM required, no tokens spent.
def _violation_knowledge_search_requires_subagent(agents_called, tool_names, query_scope):
    """knowledge_search ⇒ either daily_banking_agent or hg-invest-phase2
    must be in agents_called."""
    if "knowledge_search" not in tool_names:
        return False
    agents = agents_called or []
    return ("daily_banking_agent" not in agents) and (HG_INVEST_AGENT_NAME not in agents)


def _violation_rerank_requires_knowledge_search(agents_called, tool_names, query_scope):
    """If the `rerank` node was reached, knowledge_search must have been called."""
    if "rerank" not in (agents_called or []):
        return False
    return "knowledge_search" not in tool_names


def _violation_rule1_has_no_tools(agents_called, tool_names, query_scope):
    """`dba_no_tools` / `hg_invest_no_tools` scope means the sub-agent answered
    without tools — tool_names must be empty."""
    if query_scope not in {"dba_no_tools", "hg_invest_no_tools"}:
        return False
    return bool(tool_names)


def _violation_empty_tools_means_main_only(agents_called, tool_names, query_scope):
    """`main_agent` with no tools ⇒ only main_agent should appear on the path."""
    if tool_names or query_scope != "main_agent":
        return False
    return set(agents_called or []) - {"main_agent"} != set()


TRACE_INVARIANT_RULES = {
    "knowledge_search_requires_subagent": _violation_knowledge_search_requires_subagent,
    "rerank_requires_knowledge_search":  _violation_rerank_requires_knowledge_search,
    "rule1_scope_has_no_tools":          _violation_rule1_has_no_tools,
    "empty_tools_imply_main_agent_only": _violation_empty_tools_means_main_only,
}


def check_trace_invariants(agents_called: list[str], tools_called: list[dict],
                             query_scope: str) -> list[str]:
    """Return the IDs of every invariant rule this trace violates (empty list = clean)."""
    tool_names = {t.get("name", "") for t in tools_called or []}
    return [
        rule_id
        for rule_id, fn in TRACE_INVARIANT_RULES.items()
        if fn(agents_called, tool_names, query_scope)
    ]


def parse_trace(tc_id: str, trace: dict) -> dict:
    info = trace.get("info") or {}
    data = trace.get("data") or {}
    spans = data.get("spans") or []
    kids = _build_span_children(spans)

    tm = info.get("trace_metadata") or {}
    ti_raw = tm.get("mlflow.traceInputs", "")
    user_query = ""
    if ti_raw:
        try:
            ti = json.loads(ti_raw)
        except json.JSONDecodeError:
            ti = {}
        msgs = ti.get("messages") or []
        if msgs and isinstance(msgs[0], list) and len(msgs[0]) >= 2:
            user_query = msgs[0][1] or ""
        elif msgs and isinstance(msgs[0], dict):
            user_query = msgs[0].get("content", "") or ""

    kb_context = extract_kb_context(spans)
    kb_version = extract_kb_version(spans)
    bot_response = extract_final_ai_content(spans)
    retrieved_text, retrieved_ids = extract_retrieved_candidates(spans)
    reranker = extract_reranker_info(spans, kids)
    reranker_span_present = bool(reranker.pop("_reranker_span_present", False))
    reranker_selected_ids_found = bool(reranker.pop("_reranker_selected_ids_found", False))
    kb_context_enum_ids = extract_reranked_enum_ids(kb_context)

    # Keep the raw reranker output in `reranker_selected_ids`, but expose only
    # IDs that made it into the final KB context as `reranked_enum_ids`. This
    # filters hallucinated IDs while preserving a real empty reranker response.
    try:
        reranker_selected_ids = json.loads(reranker.get("reranker_selected_ids") or "[]")
        if not isinstance(reranker_selected_ids, list):
            reranker_selected_ids = []
    except json.JSONDecodeError:
        reranker_selected_ids = []

    if reranker_selected_ids_found:
        if kb_context_enum_ids:
            selected_set = set(reranker_selected_ids)
            reranked_enum_ids = [eid for eid in kb_context_enum_ids if eid in selected_set]
        else:
            reranked_enum_ids = reranker_selected_ids
    elif not reranker_span_present:
        reranked_enum_ids = kb_context_enum_ids
    else:
        reranked_enum_ids = []
    agent_hashes = extract_agent_prompt_hashes(spans, kids)
    calls = extract_agent_and_tool_calls(spans)
    query_scope = classify_query_scope(calls["agents_called"], calls["tools_called"])
    trace_invariant_violations = check_trace_invariants(
        calls["agents_called"], calls["tools_called"], query_scope
    )

    if query_scope == "kb":
        retrieved_id_set = set(retrieved_ids)
        raw_id_set = set(reranker_selected_ids)
        reranker_valid_selected_ids = [eid for eid in reranker_selected_ids if eid in retrieved_id_set]
        reranker_invalid_selected_ids = [eid for eid in reranker_selected_ids if eid not in retrieved_id_set]
        reranker_unselected_context_ids = [eid for eid in kb_context_enum_ids if eid not in raw_id_set]
        reranker_selection_violations = []
        if not reranker_span_present:
            reranker_selection_violations.append("reranker_span_missing")
        elif not reranker_selected_ids_found:
            reranker_selection_violations.append("reranker_selected_ids_missing")
        if reranker_invalid_selected_ids:
            reranker_selection_violations.append("reranker_selected_unknown_ids")
        if reranker_selected_ids_found and reranker_selected_ids == [] and kb_context_enum_ids:
            reranker_selection_violations.append("reranker_empty_but_context_nonempty")
        if reranker_selected_ids_found and reranker_selected_ids and reranker_unselected_context_ids:
            reranker_selection_violations.append("reranker_context_contains_unselected_ids")
        if reranker_span_present and not reranker_selected_ids_found and kb_context_enum_ids:
            reranker_selection_violations.append("reranker_context_without_selected_ids")

        if not reranker_span_present:
            reranker_selection_status = "missing_rerank"
        elif not reranker_selected_ids_found:
            reranker_selection_status = "missing_selected_ids"
        elif reranker_selected_ids == [] and kb_context_enum_ids:
            reranker_selection_status = "fallback_context_after_empty_selection"
        elif reranker_selected_ids == []:
            reranker_selection_status = "empty_selection"
        elif reranker_invalid_selected_ids:
            reranker_selection_status = "invalid_ids"
        elif reranker_unselected_context_ids:
            reranker_selection_status = "fallback_context"
        else:
            reranker_selection_status = "ok"
    else:
        reranker_valid_selected_ids = []
        reranker_invalid_selected_ids = []
        reranker_unselected_context_ids = []
        reranker_selection_violations = []
        reranker_selection_status = "not_applicable"

    # Rerank ran but selected zero candidates. Distinct from "no rerank span
    # at all" — the latter is captured by query_scope != "kb".
    reranker_selected_empty = (
        query_scope == "kb"
        and reranker_selected_ids_found
        and reranker_selected_ids == []
    )

    assessments = {a.get("assessment_name"): a for a in (info.get("assessments") or [])}

    expected_response = (
        (assessments.get("expected_response") or {})
        .get("expectation", {})
        .get("value", "")
    )

    grading = assessments.get("answer_hg_chat_admin_gpt_grading") or {}
    expert_score = (grading.get("feedback") or {}).get("value")
    # When this scorer wasn't run, value is None. Use NaN so the column stays
    # float-typed in pandas — otherwise an all-None column lands as object dtype
    # and Spark infers NullType, which gets dropped under overwriteSchema=true.
    if expert_score is None:
        expert_score = float("nan")
    expert_rationale = grading.get("rationale", "") or ""

    ter = (assessments.get("target_enums_to_relevance") or {}).get("expectation", {})
    ter_value_str = (ter.get("serialized_value") or {}).get("value", "{}")
    try:
        expected_enums_weights = json.loads(ter_value_str)
    except (json.JSONDecodeError, TypeError):
        expected_enums_weights = {}

    er2 = assessments.get("enum_relevance_2") or {}
    enum_relevance_score = (er2.get("feedback") or {}).get("value")
    if enum_relevance_score is None:
        enum_relevance_score = float("nan")

    rec = {
        "test_case_id": tc_id,
        "trace_id": info.get("trace_id"),
        "request_time": info.get("request_time"),
        "execution_duration_ms": info.get("execution_duration_ms"),
        "user_query": user_query,
        "reranked_kb_context": kb_context,
        "kb_version": kb_version,
        "reranked_enum_ids": reranked_enum_ids,
        "retrieved_candidates_text": retrieved_text,
        "retrieved_enum_ids": retrieved_ids,
        "bot_response": bot_response,
        "expected_response": expected_response,
        "expected_enums": sorted(expected_enums_weights.keys()),
        "expected_enums_weights": ter_value_str,
        "expert_score": expert_score,
        "expert_rationale": expert_rationale,
        "enum_relevance_score": enum_relevance_score,
        "agents_called": calls["agents_called"],
        "agents_only": calls["agents_only"],
        "last_agent": calls["last_agent"],
        "tools_called": calls["tools_called"],
        "query_scope": query_scope,
        "trace_invariant_violations": trace_invariant_violations,
        "reranker_selected_empty": reranker_selected_empty,
        "reranker_raw_selected_ids": json.dumps(reranker_selected_ids, ensure_ascii=False),
        "reranker_valid_selected_ids": json.dumps(reranker_valid_selected_ids, ensure_ascii=False),
        "reranker_invalid_selected_ids": json.dumps(reranker_invalid_selected_ids, ensure_ascii=False),
        "reranker_unselected_context_ids": json.dumps(reranker_unselected_context_ids, ensure_ascii=False),
        "reranker_selection_status": reranker_selection_status,
        "reranker_selection_violations": json.dumps(reranker_selection_violations, ensure_ascii=False),
    }
    rec.update(reranker)
    rec.update(agent_hashes)
    return rec


## Load JSON and parse every trace

In [0]:
with open(SKKB_JSON_PATH) as f:
    raw = json.load(f)

print(f"metadata: {raw.get('metadata')}")
print(f"trace count: {len(raw.get('traces', {}))}")

records: list[dict] = []
parse_errors: list[tuple[str, str]] = []
for tc_id, trace in raw["traces"].items():
    try:
        records.append(parse_trace(tc_id, trace))
    except Exception as exc:
        parse_errors.append((tc_id, repr(exc)))

print(f"parsed OK: {len(records)}   parse errors: {len(parse_errors)}")
if parse_errors:
    for tc_id, err in parse_errors[:10]:
        print(f"  {tc_id}: {err}")


metadata: {'source': 'tmp/results_SK_513.json', 'exported_at': '2026-04-17T11:16:21.095784+02:00', 'mlflow_tracking_uri': 'http://127.0.0.1:5000', 'total_results': 513, 'traces_exported': 513, 'traces_skipped': 0}
trace count: 513
parsed OK: 513   parse errors: 0


## Data-quality checks

In [0]:
df = pd.DataFrame.from_records(records)
print(f"rows: {len(df)}   cols: {len(df.columns)}")

missing = {
    "user_query_empty":              int((df["user_query"].str.len() == 0).sum()),
    "bot_response_empty":            int((df["bot_response"].str.len() == 0).sum()),
    "expected_response_empty":       int((df["expected_response"].str.len() == 0).sum()),
    "reranked_kb_context_empty":    int((df["reranked_kb_context"].str.len() == 0).sum()),
    "expected_enums_empty":          int(df["expected_enums"].apply(lambda x: len(x) == 0).sum()),
    "reranked_enum_ids_empty":         int(df["reranked_enum_ids"].apply(lambda x: len(x) == 0).sum()),
    "retrieved_candidates_text_empty":   int((df["retrieved_candidates_text"].str.len() == 0).sum()),
    "expert_score_null":             int(df["expert_score"].isna().sum()),
    "enum_relevance_null":           int(df["enum_relevance_score"].isna().sum()),
    "reranker_missing":              int((df["reranker_model"].str.len() == 0).sum()),
    "reranker_selected_empty_flag":  int(df["reranker_selected_empty"].sum()),
    "reranker_invalid_selected_ids": int(df["reranker_invalid_selected_ids"].apply(lambda x: len(json.loads(x)) > 0).sum()),
    "reranker_selection_violations": int(df["reranker_selection_violations"].apply(lambda x: len(json.loads(x)) > 0).sum()),
    "no_tools_called":               int(df["tools_called"].apply(lambda x: len(x) == 0).sum()),
}
for k, v in missing.items():
    print(f"  {k}: {v}")

print()
print("query_scope distribution (programmatic routing classification):")
for scope, n in df["query_scope"].value_counts().items():
    print(f"  {n:4d}  {scope}")

print()
print("agents_called value distribution (unique paths through the graph):")
_paths = df["agents_called"].apply(lambda v: " -> ".join(v) if v else "(empty)").value_counts()
for path, n in _paths.head(10).items():
    print(f"  {n:4d}  {path}")
if len(_paths) > 10:
    print(f"  ... ({len(_paths) - 10} more paths)")

print()
print("tools_called — tool-name frequency:")
from collections import Counter
_tool_counter: Counter = Counter()
for tlist in df["tools_called"]:
    for t in tlist or []:
        _tool_counter[t.get("name", "?")] += 1
for name, n in _tool_counter.most_common():
    print(f"  {n:4d}  {name}")

print()
print("reranker_selection_status distribution:")
for status, n in df["reranker_selection_status"].value_counts().items():
    print(f"  {n:4d}  {status}")

print()
print("agent-prompt fingerprint distribution (should be 1 dominant non-empty hash per agent):")
for col in ("main_agent_prompt_hash", "daily_banking_agent_prompt_hash"):
    print(f"  {col}:")
    dist = df[col].value_counts(dropna=False)
    for h, n in dist.head(5).items():
        tag = "  ← empty-string hash (agent never invoked)" if h == "d41d8cd98f" else ""
        print(f"    {h}: {n}{tag}")
    if len(dist) > 5:
        print(f"    ... ({len(dist) - 5} more hashes, total {len(dist)})")


rows: 513   cols: 35
  user_query_empty: 29
  bot_response_empty: 0
  expected_response_empty: 0
  reranked_kb_context_empty: 96
  expected_enums_empty: 0
  reranked_enum_ids_empty: 109
  retrieved_candidates_text_empty: 96
  expert_score_null: 0
  enum_relevance_null: 0
  reranker_missing: 96
  reranker_selected_empty_flag: 13
  reranker_invalid_selected_ids: 1
  reranker_selection_violations: 14
  no_tools_called: 81

query_scope distribution (programmatic routing classification):
   417  kb
    68  main_agent
    15  banking_data
    13  rule1_violation

agents_called value distribution (unique paths through the graph):
   417  main_agent -> daily_banking_agent -> llm -> tools -> retrieve -> rerank
    68  main_agent
    15  main_agent -> daily_banking_agent -> llm -> tools
    13  main_agent -> daily_banking_agent -> llm

tools_called — tool-name frequency:
   417  knowledge_search
     6  mock_banking_search_transactions
     5  mock_banking_get_accounts
     5  mock_banking_get_c

## View Data

In [0]:
#df[df.test_case_id == "Test case 33"] #["retrieved_enum_ids"])

## Debug: ENUM header-parser coverage on `reranked_kb_context`

`reranked_enum_ids` fallback extraction reads `ENUM_ID: ...` block headers from the `knowledge_search` tool output, supporting both `SAVING@...` IDs and action-style IDs such as `DISPLAY_LIMITS`. If that returns `[]` while `reranked_kb_context` is non-empty, this cell surfaces the offenders so we can inspect the trace shape.

In [0]:
_miss = df[
    (df["reranked_enum_ids"].apply(len) == 0)
    & (df["reranked_kb_context"].str.len() > 0)
].copy()

print(f"rows with non-empty reranked_kb_context but 0 parsed enums: {len(_miss)}")

# Split by whether the reranker actually selected anything.
_miss_rerank_empty = _miss[_miss["reranker_selected_empty"]]
_miss_rerank_nonempty = _miss[~_miss["reranker_selected_empty"]]
print(f"  of which, rerank-selected []  (expected — tool returned 'no matches'): {len(_miss_rerank_empty)}")
print(f"  of which, rerank-selected ≥1  (likely parser-format gap to investigate): {len(_miss_rerank_nonempty)}")

if len(_miss_rerank_nonempty):
    print()
    print("--- first 3 rerank-nonempty parser misses (first 400 chars of context) ---")
    for i, (_, r) in enumerate(_miss_rerank_nonempty.head(3).iterrows()):
        print(f"\n[{i+1}] test_case_id={r['test_case_id']}  reranker_selected_ids={r['reranker_selected_ids']}")
        print(f"    reranked_kb_context[:400]:")
        print("    " + r["reranked_kb_context"][:400].replace("\n", "\n    "))


rows with non-empty reranked_kb_context but 0 parsed enums: 13
  of which, rerank-selected []  (expected — tool returned 'no matches'): 13
  of which, rerank-selected ≥1  (likely parser-format gap to investigate): 0


## Language-mismatch scan (bot responses)

The dataset is Slovak. If the bot response contains Czech-specific characters (`ř`, `ě`) or Czech phrasing (e.g. `spořitelna`), the bot fell back into Czech — that is a language-mismatch bug the judge's rubric should flag via the `language_compliance` dimension (score 0) and the `non_slovak_phrases` evidence list. This cell surfaces the rate for awareness; it does not drop rows.

In [0]:
CZ_ONLY = re.compile(r"[řěŘĚ]|spořitelna|Česká")

def _cz_flag(s: str) -> bool:
    return isinstance(s, str) and bool(CZ_ONLY.search(s))

cz_rate = {
    "user_query_has_czech":         int(df["user_query"].apply(_cz_flag).sum()),
    "bot_response_has_czech":       int(df["bot_response"].apply(_cz_flag).sum()),
    "expected_response_has_czech":  int(df["expected_response"].apply(_cz_flag).sum()),
}
for k, v in cz_rate.items():
    print(f"  {k}: {v} / {len(df)}")


  user_query_has_czech: 0 / 513
  bot_response_has_czech: 51 / 513
  expected_response_has_czech: 1 / 513


## Drop rows missing mandatory judge inputs

We drop rows where **user_query**, **bot_response** or **expected_response** is empty — the judge cannot score those. Rows with missing KB context / reranker trace (fallback path) are kept and will typically score low on groundedness or selection dimensions; this is valid signal about retrieval quality.


In [0]:
drop_mask = (
    (df["user_query"].str.len() == 0)
    | (df["bot_response"].str.len() == 0)
    | (df["expected_response"].str.len() == 0)
)

if DROP_INCOMPLETE:
    dropped_ids = df.loc[drop_mask, "test_case_id"].tolist()
    df = df.loc[~drop_mask].copy()
    print(f"dropped {len(dropped_ids)} rows with missing mandatory fields")
    print(f"  first 10 dropped test_case_ids: {dropped_ids[:10]}")
else:
    print(f"DROP_INCOMPLETE is False; keeping {int(drop_mask.sum())} rows with missing mandatory fields")

print(f"final row count: {len(df)}")


dropped 29 rows with missing mandatory fields
  first 10 dropped test_case_ids: ['Test case 163', 'Test case 227', 'Test case 228', 'Test case 236', 'Test case 320', 'Test case 321', 'Test case 326', 'Test case 330', 'Test case 334', 'Test case 338']
final row count: 484


## Enrich: EN user query, categories, and EN/SK KB text for retrieved ENUMs

Pulls `user_query_en` + `categories_list` per test case from `test_set_513_*.json` (so no LLM translation is needed for the query) and attaches the canonical SK + EN KB text (description only) for every ENUM in `reranked_enum_ids` and `retrieved_enum_ids` using the pipe-separated `KB_GAI_SK_{SK,EN}_*.csv` dumps. Also runs a sanity check that every retrieved ENUM appears in the production `reranked_kb_context` blob.

In [0]:
# ── Enrichment sources ───────────────────────────────────────────────────────────────────
# test_set_513_*.json  — keyed by test_case_id, carries a human-authored
#                        `user_query_en` (skip LLM translation of the query)
#                        and a `categories_list` tag set per case.
# KB_GAI_SK_{SK,EN}_*.csv — pipe-separated KB dumps with columns
#                        kb.knowledgeId | kb.summary | kb.description  (summary unused — description-only).
if RUN_IN_DBX:
    # _TEST_SET_JSON = "/Workspace/Repos/Shared_HeyGeorge/ds-evals/input/kb_sk/test_set_513_2026-04-16_14h54_GAI_SK_SK.json"
    # _KB_SK_CSV     = "/Workspace/Repos/Shared_HeyGeorge/ds-evals/input/kb_sk/KB_GAI_SK_SK_2026-04-20_14h16_phase_1_2.csv"
    # _KB_EN_CSV     = "/Workspace/Repos/Shared_HeyGeorge/ds-evals/input/kb_sk/KB_GAI_SK_EN_2026-04-20_14h16_phase_1_2.csv"
    _TEST_SET_JSON = "../input/test_set_513_2026-04-16_14h54_GAI_SK_SK.json"
    _KB_SK_CSV = "../input/KB_GAI_SK_SK_2026-04-20_14h16_phase_1_2.csv"
    _KB_EN_CSV     = "../input/KB_GAI_SK_EN_2026-04-20_14h16_phase_1_2.csv"
else:
    _TEST_SET_JSON = "../../input/kb_sk/test_set_513_2026-04-16_14h54_GAI_SK_SK.json"
    _KB_SK_CSV     = "../../input/kb_sk/KB_GAI_SK_SK_2026-04-20_14h16_phase_1_2.csv"
    _KB_EN_CSV     = "../../input/kb_sk/KB_GAI_SK_EN_2026-04-20_14h16_phase_1_2.csv"

with open(_TEST_SET_JSON) as f:
    _ts = json.load(f)

_uq_en = {tid: (v.get("user_query_en") or "") for tid, v in _ts.items()}
_cats  = {tid: (v.get("categories_list") or []) for tid, v in _ts.items()}

df["user_query_en"]   = df["test_case_id"].map(_uq_en).fillna("")
df["categories_list"] = df["test_case_id"].map(lambda tid: _cats.get(tid, []))

_missing_uq = int((df["user_query_en"].str.len() == 0).sum())
print(f"user_query_en populated from test-set JSON: "
      f"{len(df) - _missing_uq}/{len(df)} (missing in {_missing_uq})")
print(f"categories_list populated: "
      f"{int(df['categories_list'].apply(bool).sum())}/{len(df)}")

# ── KB text lookup (SK + EN) ────────────────────────────────────────────────────────────
_kb_sk = pd.read_csv(_KB_SK_CSV, sep="|", dtype=str).fillna("")
_kb_en = pd.read_csv(_KB_EN_CSV, sep="|", dtype=str).fillna("")
_sk_by_id = _kb_sk.set_index("kb.knowledgeId").to_dict(orient="index")
_en_by_id = _kb_en.set_index("kb.knowledgeId").to_dict(orient="index")
print(f"KB rows  SK: {len(_kb_sk)}   EN: {len(_kb_en)}   "
      f"only-in-SK: {len(set(_sk_by_id) - set(_en_by_id))}   "
      f"only-in-EN: {len(set(_en_by_id) - set(_sk_by_id))}")


def _attach_kb(ids, lang_table):
    """Return one dict per enum id with its KB text in the requested language.

    `missing=True` means the id was not present in that language's CSV — a
    signal that either our KB dump is stale relative to what prod served, or
    the prod pipeline used a different KB version."""
    out = []
    for eid in ids or []:
        row = lang_table.get(eid) or {}
        out.append({
            "enum_id":     eid,
            "description": row.get("kb.description", ""),
            "missing":     eid not in lang_table,
        })
    return out


df["reranked_enums_kb_sk"]       = df["reranked_enum_ids"].apply(lambda ids: _attach_kb(ids, _sk_by_id))
df["reranked_enums_kb_en"]       = df["reranked_enum_ids"].apply(lambda ids: _attach_kb(ids, _en_by_id))
df["retrieved_candidates_kb_sk"] = df["retrieved_enum_ids"].apply(lambda ids: _attach_kb(ids, _sk_by_id))
df["retrieved_candidates_kb_en"] = df["retrieved_enum_ids"].apply(lambda ids: _attach_kb(ids, _en_by_id))

# ── Alignment sanity check ────────────────────────────────────────────────────────────────
# Every enum id in `reranked_enum_ids` should appear literally inside the
# `reranked_kb_context` blob that knowledge_search returned to the agent.
# A gap means prod saw text we don't have in our KB dump, or vice versa.
def _align(row):
    ctx = row["reranked_kb_context"] or ""
    ids = row["reranked_enum_ids"] or []
    missing = [i for i in ids if i not in ctx]
    kb_missing_sk = [d["enum_id"] for d in row["reranked_enums_kb_sk"] if d["missing"]]
    kb_missing_en = [d["enum_id"] for d in row["reranked_enums_kb_en"] if d["missing"]]
    return pd.Series({
        "n_enums":             len(ids),
        "n_not_in_context":    len(missing),
        "n_missing_in_sk_csv": len(kb_missing_sk),
        "n_missing_in_en_csv": len(kb_missing_en),
    })


_align_df = df.apply(_align, axis=1)
print("\nalignment — reranked_enum_ids vs reranked_kb_context + KB CSVs:")
print(f"  rows with every enum found in context:       "
      f"{int((_align_df['n_not_in_context'] == 0).sum())}/{len(df)}")
print(f"  rows with some enum missing from SK CSV:    "
      f"{int((_align_df['n_missing_in_sk_csv'] > 0).sum())}")
print(f"  rows with some enum missing from EN CSV:    "
      f"{int((_align_df['n_missing_in_en_csv'] > 0).sum())}")
print(f"  totals — lookups: {int(_align_df['n_enums'].sum())}   "
      f"missing_in_SK_csv: {int(_align_df['n_missing_in_sk_csv'].sum())}   "
      f"missing_in_EN_csv: {int(_align_df['n_missing_in_en_csv'].sum())}")

print("\nWith `user_query_en` now sourced from the test-set JSON, the"
      " translation cell only needs to cover `expected_response` and"
      " `bot_response` — not the query.")


user_query_en populated from test-set JSON: 403/484 (missing in 81)
categories_list populated: 484/484
KB rows  SK: 164   EN: 163   only-in-SK: 1   only-in-EN: 0

alignment — reranked_enum_ids vs reranked_kb_context + KB CSVs:
  rows with every enum found in context:       484/484
  rows with some enum missing from SK CSV:    0
  rows with some enum missing from EN CSV:    6
  totals — lookups: 679   missing_in_SK_csv: 0   missing_in_EN_csv: 6

With `user_query_en` now sourced from the test-set JSON, the translation cell only needs to cover `expected_response` and `bot_response` — not the query.


## Translate `expected_response` and `bot_response` to English (two persistent stores)

`user_query_en` is already populated verbatim from `test_set_513_*.json` in the enrichment cell above. The two remaining Slovak-only free-text fields are handled by two different persistent stores because their lifecycles differ:

- **`expected_response_en`** — written directly into the test-set JSON (`_TEST_SET_JSON`) alongside the existing `user_query_en` field, keyed by `test_case_id`. Expected responses are human-authored and part of the test set itself, so the test-set file is their authoritative home. A row is (re-)translated only when its `expected_response_en` entry is missing or empty.
- **`bot_response_en`** — written to a sidecar cache file next to the input traces, named `<traces-stem>_SK_EN.json` (for `traces_SK_513.json` → `traces_SK_513_SK_EN.json`). Shape: `{ "bot_response": {sk_string: en_string, ...} }`. Bot responses come from production traces; traces may get re-exported or re-parsed many times without the response text actually changing, so the cache is keyed by the **string content** and survives trace re-imports and post-processing churn.

On every notebook run, for each of the two stores:

1. Load the store (missing file or missing entries are treated as empty).
2. Translate **only** the strings that are not already present in it.
3. Merge new translations back and save atomically to the same path.

So each unique Slovak source string gets translated exactly once across Databricks runs. In local mode (`RUN_IN_DBX = False`), the notebook never imports `hg_ds_evals` and never calls the LLM — translations are expected to already live under `input/kb_sk/` and are only appended as columns.

These columns are **not** sent to the judge; they are convenience columns for the results viewer and spot-checking.


**Note**:
The code uses a cache-first strategy — it checks for existing translations before calling the LLM. Specifically:

For expected_response_en: It reads _TEST_SET_JSON, iterates over every test case, and only collects rows where expected_response_en is empty or missing into _missing_rows. Only those missing entries are sent to the GPT-5 translation call. Results are written back into the same JSON file.

For bot_response_en: It loads a sidecar JSON cache (_BOT_TR_CACHE_PATH). It compares all unique bot_response strings against the cache and builds _bot_missing — only those uncached strings are translated. The updated cache is then persisted.

In the else branch (when TRANSLATE=False or running locally): no LLM calls are made at all — it only reads whatever translations already exist on disk and maps them onto the dataframe.

So it never recomputes translations that are already cached; it only fills in the gaps.

In [0]:
from pathlib import Path as _Path
import json as _json

TRANSLATE = True
TRANSLATE_MODEL = "gpt-5-1"
TRANSLATE_MAX_CONCURRENCY = 20
TRANSLATE_MAX_OUTPUT_TOKENS = 1024
_LOCAL_TRANSLATIONS_ONLY = not RUN_IN_DBX

TRANSLATE_SYSTEM_PROMPT = (
    "You are a professional Slovak-to-English translator. Translate the "
    "user message literally and concisely. Preserve banking terminology, "
    "product names, numbers, and formatting. Do NOT add commentary, "
    "quotes, or explanations. Output ONLY the English translation."
)

# expected_response_en lives inside the test-set JSON (per test_case_id).
# bot_response_en lives in a sidecar next to the traces, keyed by string content.
_BOT_TR_CACHE_PATH = str(_Path(SKKB_JSON_PATH).with_name(_Path(SKKB_JSON_PATH).stem + "_SK_EN.json"))

def _atomic_write_json(path: str, obj) -> None:
    pp = _Path(path)
    pp.parent.mkdir(parents=True, exist_ok=True)
    tmp = pp.with_suffix(pp.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as _f:
        _json.dump(obj, _f, ensure_ascii=False, indent=2, sort_keys=True)
    tmp.replace(pp)

def _load_bot_cache(path: str) -> dict:
    try:
        with open(path, "r", encoding="utf-8") as _f:
            data = _json.load(_f)
    except FileNotFoundError:
        return {"bot_response": {}}
    if not isinstance(data, dict):
        return {"bot_response": {}}
    return {"bot_response": dict(data.get("bot_response") or {})}

if RUN_IN_DBX and TRANSLATE:
    import asyncio
    from openai import AsyncOpenAI

    _dbx_host = spark.conf.get("spark.databricks.workspaceUrl")
    _dbx_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

    _tr_client = AsyncOpenAI(
        base_url=f"https://{_dbx_host}/serving-endpoints",
        api_key=_dbx_token,
    )
    _tr_sem = asyncio.Semaphore(TRANSLATE_MAX_CONCURRENCY)

    async def _translate_one(text: str) -> str:
        if not isinstance(text, str) or not text.strip():
            return ""
        async with _tr_sem:
            resp = await _tr_client.chat.completions.create(
                model=TRANSLATE_MODEL,
                messages=[
                    {"role": "system", "content": TRANSLATE_SYSTEM_PROMPT},
                    {"role": "user",   "content": text},
                ],
                max_tokens=TRANSLATE_MAX_OUTPUT_TOKENS,
            )
        return (resp.choices[0].message.content or "").strip()

    # ── 1. expected_response_en  → written into the test-set JSON ────────
    print(f"expected_response → persisted in the test-set JSON")
    print(f"  file: {_TEST_SET_JSON}")
    with open(_TEST_SET_JSON, "r", encoding="utf-8") as _f:
        _ts_full = _json.load(_f)

    _missing_rows = []
    for _tid, _sk_text in zip(df["test_case_id"].tolist(), df["expected_response"].tolist()):
        if not isinstance(_sk_text, str) or not _sk_text.strip():
            continue
        _entry = _ts_full.get(_tid) or {}
        if not (_entry.get("expected_response_en") or ""):
            _missing_rows.append((_tid, _sk_text))

    print(f"  test-set entries missing expected_response_en: {len(_missing_rows)}")
    if _missing_rows:
        _uniq_sk = sorted({sk for _, sk in _missing_rows})
        print(f"  unique sk strings to translate: {len(_uniq_sk)}")
        _results = await asyncio.gather(*[_translate_one(s) for s in _uniq_sk])
        _sk_to_en = dict(zip(_uniq_sk, _results))
        for _tid, _sk in _missing_rows:
            if _tid not in _ts_full:
                _ts_full[_tid] = {}
            _ts_full[_tid]["expected_response_en"] = _sk_to_en.get(_sk, "")
        _atomic_write_json(_TEST_SET_JSON, _ts_full)
        print(f"  saved {len(_missing_rows)} translations → {_TEST_SET_JSON}")

    df["expected_response_en"] = df["test_case_id"].map(
        lambda tid: (_ts_full.get(tid) or {}).get("expected_response_en", "")
    )

    # ── 2. bot_response_en  → sidecar cache keyed by string content ──────
    print(f"\nbot_response → sidecar cache")
    print(f"  file: {_BOT_TR_CACHE_PATH}")
    _bot_cache = _load_bot_cache(_BOT_TR_CACHE_PATH)
    _bot_map = _bot_cache["bot_response"]
    print(f"  loaded: {len(_bot_map)} cached translations")

    _bot_uniq = sorted({
        t for t in df["bot_response"].tolist()
        if isinstance(t, str) and t.strip()
    })
    _bot_missing = [t for t in _bot_uniq if t not in _bot_map]
    print(f"  unique: {len(_bot_uniq)}   cached: {len(_bot_uniq) - len(_bot_missing)}   to translate: {len(_bot_missing)}")

    if _bot_missing:
        _new = await asyncio.gather(*[_translate_one(t) for t in _bot_missing])
        _bot_map.update(dict(zip(_bot_missing, _new)))
        _atomic_write_json(_BOT_TR_CACHE_PATH, {"bot_response": _bot_map})
        print(f"  saved → {_BOT_TR_CACHE_PATH}")

    df["bot_response_en"] = df["bot_response"].map(lambda t: _bot_map.get(t, ""))

    print(f"\ndone. expected_response_en non-empty: {(df['expected_response_en'].str.len() > 0).sum()}/{len(df)}")
    print(f"      bot_response_en      non-empty: {(df['bot_response_en'].str.len() > 0).sum()}/{len(df)}")
else:
    # Local runs only append translations already present in input/kb_sk.
    # Databricks runs with TRANSLATE=False use the same cache-only path.
    try:
        with open(_TEST_SET_JSON, "r", encoding="utf-8") as _f:
            _ts_full = _json.load(_f)
    except FileNotFoundError:
        _ts_full = {}
    df["expected_response_en"] = df["test_case_id"].map(
        lambda tid: (_ts_full.get(tid) or {}).get("expected_response_en", "")
    )
    _bot_cache = _load_bot_cache(_BOT_TR_CACHE_PATH)
    _bot_map = _bot_cache["bot_response"]
    df["bot_response_en"] = df["bot_response"].map(lambda t: _bot_map.get(t, ""))
    if _LOCAL_TRANSLATIONS_ONLY:
        print("RUN_IN_DBX=False — using input/kb_sk translation files only; no hg_ds_evals import and no LLM calls.")
    else:
        print("TRANSLATE=False — using on-disk caches only, no LLM calls.")
    print(f"  expected_response_en non-empty: {(df['expected_response_en'].str.len() > 0).sum()}/{len(df)}")
    print(f"  bot_response_en      non-empty: {(df['bot_response_en'].str.len() > 0).sum()}/{len(df)}")

expected_response → persisted in the test-set JSON
  file: ../input/test_set_513_2026-04-16_14h54_GAI_SK_SK.json
  test-set entries missing expected_response_en: 0

bot_response → sidecar cache
  file: ../input/traces_SK_513_SK_EN.json
  loaded: 484 cached translations
  unique: 484   cached: 484   to translate: 0

done. expected_response_en non-empty: 484/484
      bot_response_en      non-empty: 484/484


In [0]:
print("--- Secret scopes ---")
scopes = dbutils.secrets.listScopes()
for s in scopes:
    print(s.name)

print("\n--- Secrets in Key-Vault scope ---")
keys = dbutils.secrets.list("key-vault")
for k in keys:
    print(k.key)

--- Secret scopes ---
databricks-package-management
key-vault
ss-global-key-vault

--- Secrets in Key-Vault scope ---
aiexpert-george-hu-client-id
aiexpert-george-hu-secret
AZURE-COGNITIVESERVICES-ENDPOINT
AZURE-COGNITIVESERVICES-KEY
clientid
clientsecret


## Length distribution sanity check

In [0]:
for col in ("user_query", "reranked_kb_context", "retrieved_candidates_text", "bot_response", "expected_response", "reranker_user_prompt"):
    lens = df[col].str.len()
    print(f"  {col:<25} n={len(lens)}  min={int(lens.min())}  median={int(lens.median())}  p95={int(lens.quantile(0.95))}  max={int(lens.max())}")


  user_query                n=484  min=11  median=46  p95=74  max=100
  reranked_kb_context       n=484  min=0  median=2833  p95=13970  max=46122
  retrieved_candidates_text n=484  min=0  median=23202  p95=43946  max=67817
  bot_response              n=484  min=153  median=653  p95=1756  max=3044
  expected_response         n=484  min=60  median=368  p95=809  max=1166
  reranker_user_prompt      n=484  min=0  median=24045  p95=45309  max=69152


The judge runs with `max_output_tokens: 16000`. `retrieved_candidates_text` is the heaviest input field (~20 kB p95); the YAML's `max_output_tokens` of 16000 refers to the judge's *output* budget, not input, so this is fine, but be mindful when tweaking the rubric.

## Peek at one row

In [0]:
sample = df.iloc[0].to_dict()
for k in ("reranked_kb_context", "retrieved_candidates_text", "bot_response", "expected_response", "expert_rationale", "reranker_user_prompt", "reranker_system_prompt"):
    if isinstance(sample.get(k), str) and len(sample[k]) > 300:
        sample[k] = sample[k][:300] + "..."
for k, v in sample.items():
    print(f"  {k}: {v}")


  test_case_id: Test case 1
  trace_id: tr-46162b7fdda7bf655c03f803aa217f15
  request_time: 2026-04-16T22:06:51.508Z
  execution_duration_ms: 20477
  user_query: Dá sa na sporenie vložiť aj hotovosť, alebo len prevodom?
  reranked_kb_context: [KB_GAI_SK_SK_2026-03-17_17h26]
SAVING@ACCOUNT_MOVEMENTS: **Vklady na sporiaci účet:**\nNajjednoduchšie môžete vložiť peniaze bezhotovostným prevodom -- z vášho účtu v Slovenskej sporiteľni alebo z účtu v inej banke. Prevod môžete urobiť jednorazovo alebo si nastaviť trvalý príkaz. Cez Georgea urobí...
  kb_version: KB_GAI_SK_SK_2026-03-17_17h26
  reranked_enum_ids: ['SAVING@ACCOUNT_MOVEMENTS', 'SAVING@KIDS_WITHDRAWALS_AND_DEPOSITS']
  retrieved_candidates_text: SAVING@ACCOUNT_MOVEMENTS: **Vklady na sporiaci účet:**\nNajjednoduchšie môžete vložiť peniaze bezhotovostným prevodom -- z vášho účtu v Slovenskej sporiteľni alebo z účtu v inej banke. Prevod môžete urobiť jednorazovo alebo si nastaviť trvalý príkaz. Cez Georgea urobíte prevod cez možnosť 

## Convert list-typed columns to JSON strings for Spark compatibility

In [0]:
# Delta/Parquet can natively hold arrays, but stringifying keeps the schema simple
# and matches how hg_ds_evals renders input_fields into the user prompt.
df["reranked_enum_ids"] = df["reranked_enum_ids"].apply(json.dumps)
df["expected_enums"] = df["expected_enums"].apply(json.dumps)
df["retrieved_enum_ids"] = df["retrieved_enum_ids"].apply(json.dumps)
df["categories_list"] = df["categories_list"].apply(lambda v: json.dumps(v, ensure_ascii=False))
for _c in ("reranked_enums_kb_sk", "reranked_enums_kb_en",
             "retrieved_candidates_kb_sk", "retrieved_candidates_kb_en"):
    df[_c] = df[_c].apply(lambda v: json.dumps(v, ensure_ascii=False))
df["agents_called"] = df["agents_called"].apply(lambda v: json.dumps(v, ensure_ascii=False))
df["tools_called"] = df["tools_called"].apply(lambda v: json.dumps(v, ensure_ascii=False))
# reranker_* diagnostic ID lists / reranker_token_usage are already JSON strings.

print(df.dtypes)


test_case_id                        object
trace_id                            object
request_time                        object
execution_duration_ms                int64
user_query                          object
reranked_kb_context                 object
kb_version                          object
reranked_enum_ids                   object
retrieved_candidates_text           object
retrieved_enum_ids                  object
bot_response                        object
expected_response                   object
expected_enums                      object
expected_enums_weights              object
expert_score                       float64
expert_rationale                    object
enum_relevance_score               float64
agents_called                       object
tools_called                        object
query_scope                         object
trace_invariant_violations          object
reranker_selected_empty               bool
reranker_raw_selected_ids           object
reranker_va

## Write Delta table

In [0]:
if RUN_IN_DBX:
    # Serialise trace_invariant_violations (may be all-None → NullType in ArrayType,
    # which Delta rejects).
    if "trace_invariant_violations" in df.columns:
        df["trace_invariant_violations"] = df["trace_invariant_violations"].apply(
            lambda v: json.dumps(v, ensure_ascii=False) if isinstance(v, list) else (v if isinstance(v, str) else "[]")
        )

    sdf = spark.createDataFrame(df)

    full_table = f"{DBX_CATALOG}.{DBX_SCHEMA}.{DBX_TABLE}"
    (
        sdf
        .write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table)
    )

    print(f"wrote {sdf.count()} rows to {full_table}")

wrote 484 rows to prod_aut_chat00_lab_catalog.private_uc0115_aix_db.dts_eval_skkb_exp_001


## Verify table

In [0]:
if RUN_IN_DBX:
    rb = spark.read.table(f"{DBX_CATALOG}.{DBX_SCHEMA}.{DBX_TABLE}")
    print(f"read back row count: {rb.count()}")
    rb.printSchema()
    rb.select(
        "test_case_id",
        "query_scope",
        "user_query",
        "user_query_en",
        "expert_score",
        "enum_relevance_score",
        "reranker_model",
        "reranker_selected_ids",
        "reranker_selection_status",
        "reranker_invalid_selected_ids",
        "reranker_unselected_context_ids",
        "reranker_selected_empty",
        "retrieved_enum_ids",
        "agents_called",
        "tools_called",
    ).show(5, truncate=120)


read back row count: 484
root
 |-- test_case_id: string (nullable = true)
 |-- trace_id: string (nullable = true)
 |-- request_time: string (nullable = true)
 |-- execution_duration_ms: long (nullable = true)
 |-- user_query: string (nullable = true)
 |-- reranked_kb_context: string (nullable = true)
 |-- kb_version: string (nullable = true)
 |-- reranked_enum_ids: string (nullable = true)
 |-- retrieved_candidates_text: string (nullable = true)
 |-- retrieved_enum_ids: string (nullable = true)
 |-- bot_response: string (nullable = true)
 |-- expected_response: string (nullable = true)
 |-- expected_enums: string (nullable = true)
 |-- expected_enums_weights: string (nullable = true)
 |-- expert_score: double (nullable = true)
 |-- expert_rationale: string (nullable = true)
 |-- enum_relevance_score: double (nullable = true)
 |-- agents_called: string (nullable = true)
 |-- tools_called: string (nullable = true)
 |-- query_scope: string (nullable = true)
 |-- trace_invariant_violations

In [0]:
rb.display()

In [0]:
cols = ["user_query",
        "user_query_en",
        "expert_score",
        "enum_relevance_score",
        "reranker_model",
        "reranker_selected_ids",
        "reranker_selection_status",
        "reranker_invalid_selected_ids",
        "reranker_unselected_context_ids",
        "reranker_selected_empty",
        "retrieved_enum_ids",
        "agents_called",
        "tools_called"]
df[df.test_case_id == "Test case 33"][cols]

In [0]:
# Retrieved documents should never be empty
from numpy import dtype
print(df.retrieved_enum_ids.dtype)
print(sum(pd.isna(df.retrieved_enum_ids)))
print(sum(df.retrieved_enum_ids.apply(lambda x: x == "[]")))
print(sum(df.retrieved_enum_ids.apply(lambda x: x == 0)))
print(sum(df.retrieved_enum_ids.apply(lambda x: len(x) <= 3)))

In [0]:
df[df.retrieved_enum_ids.apply(lambda x: len(x) <= 3)][["test_case_id", "retrieved_enum_ids"]]

In [0]:
df.reranker_selection_status.value_counts(dropna=False)